# Test Granite Tiny Polish Capabilities

**How to use this notebook:**
1. Go to [Google Colab](https://colab.research.google.com)
2. Click File → Upload notebook
3. Upload this file: `06-test_granite_polish.ipynb`
4. Select Runtime → Change runtime type → GPU (T4/V100/A100)
5. Run all cells

This notebook tests Granite Tiny model on Polish language tasks **before and after fine-tuning**.

**Usage:**
1. Run this notebook BEFORE fine-tuning to get baseline results
2. Fine-tune the model using `granite_tiny_polish_finetuning_pro.ipynb`
3. Run this notebook AFTER fine-tuning to compare results

**Requirements:**
- Google Colab (Free or Pro)
- GPU runtime (T4, V100, or A100)

# notes

<!-- If crashes persist:

Restart runtime completely
Reduce max_seq_length from 2048 to 1024
Reduce max_new_tokens from 512 to 256
Use T4/V100/A100 GPU (not CPU) -->

## Step 1: Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 2: Install Dependencies

In [ ]:
# Install Unsloth and let it handle PyTorch dependencies
!pip install --upgrade --no-cache-dir unsloth
!pip install "transformers<=5.5.0" accelerate bitsandbytes

print("✓ Installation complete!")
print("\n⚠ IMPORTANT: Restarting runtime automatically...")
print("After restart, continue from Step 3.")

# Auto-restart runtime to reload all modules
import os
os.kill(os.getpid(), 9)

**⚠ Runtime will restart automatically after Step 2**

The notebook will restart the runtime automatically to ensure all packages load correctly.

After restart, **continue from Step 3 below**.

## Step 3: Get Test Questions

**Choose ONE option:**

### Option A: Create Test Questions in Colab (Recommended - No Upload Needed)

In [ ]:
import json

# Create test questions directly in Colab
test_questions = [
    {"id": 1, "category": "grammar", "instruction": "Popraw błędy gramatyczne w zdaniu.", "input": "Wczoraj poszedłem do sklepu i kupiłem jabłka, gruszki i bananów.", "expected_output": "Wczoraj poszedłem do sklepu i kupiłem jabłka, gruszki i banany."},
    {"id": 2, "category": "translation", "instruction": "Przetłumacz na polski.", "input": "The quick brown fox jumps over the lazy dog.", "expected_output": "Szybki brązowy lis przeskakuje przez leniwego psa."},
    {"id": 3, "category": "summarization", "instruction": "Podsumuj tekst w jednym zdaniu.", "input": "Polska jest krajem położonym w Europie Środkowej. Graniczy z Niemcami, Czechami, Słowacją, Ukrainą, Białorusią, Litwą i Rosją. Stolicą Polski jest Warszawa, a język urzędowy to polski.", "expected_output": "Polska to kraj w Europie Środkowej graniczący z siedmioma państwami, ze stolicą w Warszawie i językiem polskim jako urzędowym."},
    {"id": 4, "category": "math", "instruction": "Rozwiąż zadanie matematyczne.", "input": "Jeśli jabłko kosztuje 2 zł, a gruszka 3 zł, ile zapłacę za 5 jabłek i 3 gruszki?", "expected_output": "Za 5 jabłek zapłacisz 10 zł (5 × 2 zł), a za 3 gruszki 9 zł (3 × 3 zł). Razem: 10 zł + 9 zł = 19 zł."},
    {"id": 5, "category": "classification", "instruction": "Sklasyfikuj ton wiadomości jako: pozytywny, negatywny lub neutralny.", "input": "Dziękuję za wspaniałą obsługę! Wszystko przebiegło sprawnie i jestem bardzo zadowolony.", "expected_output": "pozytywny"},
    {"id": 6, "category": "question_answering", "instruction": "Odpowiedz na pytanie na podstawie tekstu.", "input": "Tekst: Jan Kowalski urodził się w 1985 roku w Krakowie. Studiował informatykę na Politechnice Warszawskiej. Pytanie: Gdzie studiował Jan Kowalski?", "expected_output": "Jan Kowalski studiował na Politechnice Warszawskiej."},
    {"id": 7, "category": "creative", "instruction": "Napisz krótki wiersz o jesieni (4 wersy).", "input": "", "expected_output": "Złote liście spadają z drzew,\nWiatr szumi w koronach cicho,\nJesień maluje świat na nowo,\nPrzyroda zasypia spokojnie."},
    {"id": 8, "category": "reasoning", "instruction": "Rozwiąż zagadkę logiczną.", "input": "Mam 3 siostry. Każda z moich sióstr ma 3 siostry. Ile sióstr jest w sumie?", "expected_output": "W sumie są 4 siostry (ty i twoje 3 siostry). Każda z was ma 3 siostry, co daje razem 4 osoby."},
    {"id": 9, "category": "formatting", "instruction": "Sformatuj adres w standardowy sposób.", "input": "jan kowalski ul nowak 15 00-001 warszawa polska", "expected_output": "Jan Kowalski\nul. Nowak 15\n00-001 Warszawa\nPolska"},
    {"id": 10, "category": "extraction", "instruction": "Wyodrębnij wszystkie daty z tekstu.", "input": "Spotkanie odbędzie się 15 marca 2026 roku. Termin zgłoszeń upływa 1 lutego 2026. Wyniki zostaną ogłoszone 30 kwietnia 2026.", "expected_output": "- 15 marca 2026\n- 1 lutego 2026\n- 30 kwietnia 2026"},
    {"id": 11, "category": "comparison", "instruction": "Porównaj dwa pojęcia.", "input": "Porównaj kawę i herbatę.", "expected_output": "Kawa zawiera więcej kofeiny niż herbata i ma intensywniejszy smak. Herbata jest łagodniejsza, zawiera antyoksydanty i ma więcej odmian smakowych. Oba napoje są popularne na całym świecie."},
    {"id": 12, "category": "instruction_following", "instruction": "Napisz listę zakupów zawierającą dokładnie 5 produktów spożywczych, każdy w nowej linii.", "input": "", "expected_output": "- mleko\n- chleb\n- masło\n- jajka\n- ser"},
    {"id": 13, "category": "sentiment", "instruction": "Oceń nastrój tekstu w skali 1-10 (1=bardzo negatywny, 10=bardzo pozytywny).", "input": "Produkt jest w porządku, ale oczekiwałem czegoś lepszego za tę cenę.", "expected_output": "5 - neutralny z lekkim rozczarowaniem"},
    {"id": 14, "category": "paraphrasing", "instruction": "Przeformułuj zdanie zachowując jego znaczenie.", "input": "Deszcz pada od rana.", "expected_output": "Od rana pada deszcz."},
    {"id": 15, "category": "entity_recognition", "instruction": "Wypisz wszystkie nazwy własne z tekstu.", "input": "Anna Nowak spotkała się z Piotrem Kowalskim w Warszawie w restauracji Pod Złotym Lwem.", "expected_output": "- Anna Nowak (osoba)\n- Piotr Kowalski (osoba)\n- Warszawa (miasto)\n- Pod Złotym Lwem (restauracja)"},
    {"id": 16, "category": "definition", "instruction": "Zdefiniuj pojęcie.", "input": "Co to jest fotosynteza?", "expected_output": "Fotosynteza to proces, w którym rośliny wykorzystują światło słoneczne, wodę i dwutlenek węgla do produkcji glukozy i tlenu. Jest to podstawowy proces umożliwiający życie na Ziemi."},
    {"id": 17, "category": "advice", "instruction": "Udziel porady.", "input": "Jak mogę poprawić swoją koncentrację podczas nauki?", "expected_output": "Aby poprawić koncentrację: 1) Usuń rozpraszacze (telefon, media społecznościowe), 2) Rób regularne przerwy (technika Pomodoro), 3) Stwórz odpowiednie środowisko do nauki, 4) Wysypiaj się, 5) Podziel materiał na mniejsze części."},
    {"id": 18, "category": "error_detection", "instruction": "Znajdź błąd w obliczeniach.", "input": "15 + 23 = 48", "expected_output": "Błąd: 15 + 23 = 38, a nie 48. Prawidłowy wynik to 38."},
    {"id": 19, "category": "continuation", "instruction": "Dokończ historię (2-3 zdania).", "input": "Pewnego dnia Maria znalazła tajemniczy klucz w starym pudełku na strychu.", "expected_output": "Klucz był ozdobiony dziwnymi symbolami, których nigdy wcześniej nie widziała. Postanowiła poszukać zamka, do którego pasuje, rozpoczynając tym samym niezwykłą przygodę."},
    {"id": 20, "category": "factual", "instruction": "Odpowiedz na pytanie faktograficzne.", "input": "Jaka jest stolica Polski?", "expected_output": "Stolicą Polski jest Warszawa."}
]

# Save to file
with open('polish_test_questions.jsonl', 'w', encoding='utf-8') as f:
    for q in test_questions:
        f.write(json.dumps(q, ensure_ascii=False) + '\n')

print(f"✓ Created {len(test_questions)} test questions")
print("File saved as: polish_test_questions.jsonl")

### Option B: Upload from Local Machine (Alternative)

If you have the file locally, you can upload it instead.

In [ ]:
# Uncomment and run this cell if you want to upload the file instead
# from google.colab import files
# print("Please upload polish_test_questions.jsonl:")
# uploaded = files.upload()
# if 'polish_test_questions.jsonl' in uploaded:
#     print("✓ File uploaded successfully!")

## Step 4: Load Test Questions

In [ ]:
import json

def load_test_questions(filepath="polish_test_questions.jsonl"):
    """Load test questions from JSONL file."""
    questions = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            questions.append(json.loads(line))
    return questions

questions = load_test_questions()
print(f"✓ Loaded {len(questions)} test questions")

# Show categories
categories = set(q['category'] for q in questions)
print(f"\nCategories: {', '.join(sorted(categories))}")

## Step 5: Configure Model to Test

**Choose ONE option:**

### Option A: Test Base Model (Before Fine-tuning)

**This will download the base Granite Tiny model from HuggingFace automatically.**

No upload needed - the model (~4GB) will be downloaded when you load it in Step 6.

In [ ]:
# Option A: Base model (downloads automatically from HuggingFace)
MODEL_NAME = "unsloth/granite-4.0-h-tiny"
OUTPUT_FILE = "results_before_finetuning.json"

print(f"Testing: {MODEL_NAME}")
print(f"This model will be downloaded from HuggingFace (~4GB)")
print(f"Results will be saved to: {OUTPUT_FILE}")

### Option B: Test Fine-tuned Model (After Fine-tuning)

**You need to have your fine-tuned model available in one of these ways:**

1. **Saved to Google Drive** (recommended during training)
2. **Upload the model folder** to Colab

The fine-tuned model should be in a folder like `granite4-tiny-h-polish-lora-pro/`

In [ ]:
# Option B: Fine-tuned model
# If you saved to Google Drive:
from google.colab import drive
drive.mount('/content/drive')

MODEL_NAME = "/content/drive/MyDrive/granite4-tiny-h-polish-lora-pro"
# Or if uploaded locally:
# MODEL_NAME = "granite4-tiny-h-polish-lora-pro"

OUTPUT_FILE = "results_after_finetuning.json"

print(f"Testing: {MODEL_NAME}")
print(f"Results will be saved to: {OUTPUT_FILE}")

In [ ]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print(f"✓ GPU cache cleared")


## Step 6: Load Model

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading model: {MODEL_NAME}")
print("This may take 2-3 minutes...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Enable inference mode
FastLanguageModel.for_inference(model)

print("✓ Model loaded successfully!")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 7: Define Test Functions

In [ ]:
def format_prompt(instruction, input_text=""):
    """Format prompt using Granite's chat template."""
    user_message = instruction
    if input_text:
        user_message += f"\n\n{input_text}"
    
    prompt = f"""<|start_of_role|>user<|end_of_role|>
{user_message}
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>
"""
    return prompt


def generate_response(prompt, max_new_tokens=512, temperature=0.7):
    """Generate response from the model."""
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract assistant response
    assistant_start = response.find("<|start_of_role|>assistant<|end_of_role|>")
    if assistant_start != -1:
        assistant_response = response[assistant_start + len("<|start_of_role|>assistant<|end_of_role|>"):].strip()
        assistant_response = assistant_response.replace("<|end_of_text|>", "").strip()
        return assistant_response
    
    return response


def test_single_question(question, show_details=True):
    """Test model on a single question."""
    if show_details:
        print(f"\n{'='*60}")
        print(f"ID: {question['id']} | Category: {question['category']}")
        print(f"{'='*60}")
        print(f"Instruction: {question['instruction']}")
        if question['input']:
            print(f"Input: {question['input'][:100]}..." if len(question['input']) > 100 else f"Input: {question['input']}")
    
    # Format and generate
    prompt = format_prompt(question['instruction'], question['input'])
    
    try:
        response = generate_response(prompt)
        
        if show_details:
            print(f"\nExpected: {question['expected_output'][:150]}..." if len(question['expected_output']) > 150 else f"\nExpected: {question['expected_output']}")
            print(f"\nModel Output:\n{response}")
        
        return {
            "id": question['id'],
            "category": question['category'],
            "instruction": question['instruction'],
            "input": question['input'],
            "expected_output": question['expected_output'],
            "model_output": response,
            "success": True
        }
    except Exception as e:
        if show_details:
            print(f"\n⚠ Error: {str(e)}")
        return {
            "id": question['id'],
            "category": question['category'],
            "instruction": question['instruction'],
            "input": question['input'],
            "expected_output": question['expected_output'],
            "model_output": f"ERROR: {str(e)}",
            "success": False
        }

print("✓ Test functions defined!")

## Step 8: Run Tests

This will test all 20 questions. Takes ~5-10 minutes.

In [ ]:
from datetime import datetime

print(f"\n{'='*60}")
print(f"TESTING: {MODEL_NAME}")
print(f"{'='*60}")
print(f"Total questions: {len(questions)}")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

results = []
for i, question in enumerate(questions, 1):
    print(f"\n[{i}/{len(questions)}] Testing...")
    result = test_single_question(question, show_details=True)
    results.append(result)

print(f"\n{'='*60}")
print("✓ Testing complete!")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

## Step 9: Analyze Results

In [ ]:
# Summary statistics
total = len(results)
successful = sum(1 for r in results if r['success'])
failed = total - successful

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total questions: {total}")
print(f"Successful: {successful} ({successful/total*100:.1f}%)")
print(f"Failed: {failed} ({failed/total*100:.1f}%)")

# By category
print(f"\n{'='*60}")
print("BY CATEGORY")
print(f"{'='*60}")

categories = {}
for r in results:
    cat = r['category']
    if cat not in categories:
        categories[cat] = {'total': 0, 'success': 0}
    categories[cat]['total'] += 1
    if r['success']:
        categories[cat]['success'] += 1

for cat in sorted(categories.keys()):
    stats = categories[cat]
    rate = (stats['success'] / stats['total']) * 100
    print(f"{cat:20s}: {stats['success']}/{stats['total']} ({rate:.0f}%)")

## Step 10: Save Results

In [ ]:
output_data = {
    "model": MODEL_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_questions": total,
    "successful": successful,
    "failed": failed,
    "success_rate": successful / total,
    "results": results
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\n✓ Results saved to: {OUTPUT_FILE}")
print(f"\nFile size: {len(json.dumps(output_data, ensure_ascii=False)) / 1024:.1f} KB")

## Step 11: Download Results

In [ ]:
from google.colab import files

files.download(OUTPUT_FILE)
print(f"✓ Downloaded: {OUTPUT_FILE}")

## Step 12: Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Copy to Drive
drive_path = f"/content/drive/MyDrive/{OUTPUT_FILE}"
shutil.copy(OUTPUT_FILE, drive_path)

print(f"✓ Saved to Google Drive: {drive_path}")

## Next Steps

### If you tested the BASE model:
1. Download `results_before_finetuning.json`
2. Fine-tune the model using `granite_tiny_polish_finetuning_pro.ipynb`
3. Come back and test the fine-tuned model

### If you tested the FINE-TUNED model:
1. Download `results_after_finetuning.json`
2. Compare with `results_before_finetuning.json`
3. Analyze improvements in:
   - Polish grammar and fluency
   - Task-specific accuracy
   - Instruction following
   - Response quality

### Comparison Tips:
- Look at specific examples side-by-side
- Check if Polish responses are more natural
- Verify task completion accuracy
- Note any regressions or improvements